In [94]:
import os
import json
import cohere
from groq import Groq
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client, tracing_context

# Initialize the Cohere and Groq clients
co_client = cohere.Client()
groq_client = Groq()

In [95]:
qdrant_client = QdrantClient(url="http://localhost:6333")

## Download all data from Qdrant

In [96]:
all_points, next_page_offset = qdrant_client.scroll(
    collection_name="amazon-items-collection-00",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False,
)

In [97]:
all_points[0].payload

{'description': "BoYata Screen Cleaner Kit 4PCS, 3-in-1 Touchscreen Mist Cleaner Spray Bottle Tool & Microfiber Cloth, Portable Reusable Fingerprint-Proof Spray Bottle for Phone/Laptop/Tablet/TV/Monitor/Car Screens Not Containing Cleaning Fluid: BoYata all-in-one detachable fingerprint-proof computer screen cleaner consists of a protective case, microfiber cloth shell and spray bottle (without cleaning fluid), integrated spray and wipe in one. Extremely convenient to use after filling cleaning fluid. Note: please don't put corrosive liquids. Wide Compatibility: The touch screen mist cleaner is useful for screen cleaning. Suitable for all phones/ tablets/ laptops/ monitors/ TVs/ cars screens, glass surface etc. Spray and wipe in one for quick and easy cleaning, making your screen look as good as new. Safe and no damage to your screen. Powerful Cleaning: The updated microfiber cloth of the screen cleaner is easy to move fingerprints, dust, grease, dirt on screen without streaking or scra

In [98]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points]

In [99]:
all_context

[{'id': 'B0C3VNSX6X',
  'text': "BoYata Screen Cleaner Kit 4PCS, 3-in-1 Touchscreen Mist Cleaner Spray Bottle Tool & Microfiber Cloth, Portable Reusable Fingerprint-Proof Spray Bottle for Phone/Laptop/Tablet/TV/Monitor/Car Screens Not Containing Cleaning Fluid: BoYata all-in-one detachable fingerprint-proof computer screen cleaner consists of a protective case, microfiber cloth shell and spray bottle (without cleaning fluid), integrated spray and wipe in one. Extremely convenient to use after filling cleaning fluid. Note: please don't put corrosive liquids. Wide Compatibility: The touch screen mist cleaner is useful for screen cleaning. Suitable for all phones/ tablets/ laptops/ monitors/ TVs/ cars screens, glass surface etc. Spray and wipe in one for quick and easy cleaning, making your screen look as good as new. Safe and no damage to your screen. Powerful Cleaning: The updated microfiber cloth of the screen cleaner is easy to move fingerprints, dust, grease, dirt on screen without s

In [100]:
len(all_context)

50

### Download it for Gemini pro do it, save token for llm in this rag

In [101]:
import json

file_path = r"C:\Users\Chanreach\Documents\AI-Engineer\data\all_context.json"

with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(all_context, f, indent=4)
    
print(f"Data successfully saved to: {file_path}")

Data successfully saved to: C:\Users\Chanreach\Documents\AI-Engineer\data\all_context.json


## Render a prompt to generate synthetic Eval reference dataset 

In [102]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""


USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{all_context}
"""

In [103]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{
  "type": "array",
  "items": {
    "

In [104]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[{'id': 'B0C3VNSX6X', 'text': "BoYata Screen Cleaner Kit 4PCS, 3-in-1 Touchscreen Mist Cleaner Spray Bottle Tool & Microfiber Cloth, Portable Reusable Fingerprint-Proof Spray Bottle for Phone/Laptop/Tablet/TV/Monitor/Car Screens Not Containing Cleaning Fluid: BoYata all-in-one detachable fingerprint-proof computer screen cleaner consists of a protective case, microfiber cloth shell and spray bottle (without cleaning fluid), integrated spray and wipe in one. Extremely convenient to use after filling cleaning fluid. Note: please don't put corrosive liquids. Wide Compatibility: The touch screen mist cleaner is useful for screen cleaning. Suitable for all phones/ tablets/ laptops/ monitors/ TVs/ cars screens, glass surface etc. Spray and wipe in one for quick and easy cleaning, making your screen look as good as new. Safe and no damage to your screen. Powerful Cleaning: The updated microfiber cloth of the scre

In [105]:
# response = groq_client.chat.completions.create(
#     model="llama-3.1-8b-instant",
#     messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": USER_PROMPT}]
    
# )

# print(response.choices[0].message.content)

In [106]:
# import json

# # 1. Grab the raw text from the Gemini response
# json_output = response.text.strip()

# # 2. Strip out the markdown backticks if Gemini included them
# if json_output.startswith("```json"):
#     json_output = json_output.split("```json")[1]
# if json_output.endswith("```"):
#     json_output = json_output.rsplit("```", 1)[0]
    
# json_output = json_output.strip()

# # 3. Parse the clean JSON string into a Python dictionary/list
# json_output = json.loads(json_output)

# # Optional: Print it to verify it parsed correctly!


In [107]:
import json

# 1. Load the final 50 questions directly from the new file
file_path = r"C:\Users\Chanreach\Documents\AI-Engineer\data\evaluation_dataset_50.json"

with open(file_path, 'r', encoding='utf-8') as f:
    json_output = json.load(f)

# Optional: Print it to verify it parsed correctly!
print(f"Successfully loaded {len(json_output)} questions from Gemini!")

Successfully loaded 50 questions from Gemini!


In [108]:
json_output

[{'question': 'What smartwatches do you have available and which one has more sport modes?',
  'chunk_ids': ['B0B2R7HK3C', 'B0B8S534KB'],
  'answer_example': 'We have two smartwatches available: the Boean Smart Watch and the KOSPET Military Smart Watch. The KOSPET watch has more sport modes, offering 24 modes, while the Boean watch offers 14.',
  'reasoning': 'Chunks B0B2R7HK3C and B0B8S534KB contain the descriptions for the two smartwatches in stock.'},
 {'question': "I'm looking for a new laptop for school. What ASUS and HP models do you have in stock?",
  'chunk_ids': ['B0BK9T6YT6', 'B0BLP8DHVR', 'B0BQTX62SD', 'B09F915P41'],
  'answer_example': 'For ASUS, we offer the VivoBook 15 and the Vivobook 17.3. For HP, we have the Victus 15 Gaming Laptop and a 15.6 Slim and Light model.',
  'reasoning': 'These four chunks contain all the available ASUS and HP laptops in the catalog.'},
 {'question': 'Do you sell any USB flash drives with at least 128GB of storage space?',
  'chunk_ids': ['B0

In [109]:
len(json_output)

50

In [110]:
# Grab a valid ID from our generated questions to test with
test_id = json_output[0]['chunk_ids'][0]

points = qdrant_client.scroll(
    collection_name="amazon-items-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value=test_id) # <-- Dynamically use a valid ID
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [111]:
points[0].payload

{'description': 'Smart Watch, Fitness Tracker with 5ATM Waterproof, Heart Rate Monitor and Blood Oxygen Monitor, Sleep Monitor 1.69-inch HD Color Screen Smartwatch for Men Women Compatible with iPhone & Android 【Customized Your Unique Smart watch】Boean smart watch with 1.69-inch TFT-LCD full touch screen technology makes your operation quick and easy. You can set the watch face to show your personality with the existing images or upload images. The smart watch will vibrate when receiving incoming calls, message notifications from Facebook, Instagram, Twitter, WhatsApp, etc. Never miss any important calls or messages. 【Long Battery & 5ATM Waterproof】It only takes 2 hours to charge and can be used for 8-15 days or 30 days standby time. Boean Smart Watch is designed with 5ATM waterproof rating (50m waterproof), you can wear it while swimming or sweating out. (Note: the smartwatch does not prevent hot water and sea water). 【24/7 Heart Rate Monitor & Sleep Tracker】Advanced HR sensor for acc

In [112]:
def get_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name="amazon-items-collection-00",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin) # <--- Fixed!
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]
    
    # Safely return the description if found
    if len(points) > 0:
        return points[0].payload["description"]
    else:
        return "Description not found." 

In [113]:
get_description("B0BNVKS9WH")

'Description not found.'

## Create Eval dataset in Langsmith

In [114]:
client = Client(api_key=os.environ.get("LANGSMITH_API_KEY"))

In [115]:
dataset_name = "rag-evaluation-dataset-50"

# If the dataset already exists from a previous run, we can delete it first to start fresh
if client.has_dataset(dataset_name=dataset_name):
    client.delete_dataset(dataset_name=dataset_name)

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset of 50 questions for evaluating RAG pipeline"
)

In [116]:
for item in json_output:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )

## 12